# 🏥 Improved Kidney Disease Classification

This notebook demonstrates the improved model for kidney disease classification from CT scans.

## Key Improvements Over Original Model

| Issue in Original Model | Solution in Improved Model |
|-------------------------|-----------------------------|
| Training CNN from scratch (overfitting) | Transfer Learning with MobileNetV2 |
| Weak data augmentation | Enhanced CT-specific augmentation |
| No strong regularization | Added Dropout layers (0.5) |
| Monitoring wrong metric (val_accuracy) | Using val_loss for early stopping |
| Limited learning capability | Fine-tuning unfrozen layers |

## Setup & Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Configuration
BASE_PATH = r"C:\Users\chife\Downloads\Esprit-PIdev-4SAE11-2025-2026-NephroCare-final\Esprit-PIdev-4SAE11-2025-2026-NephroCare-final\Kidney Project IA\archive\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
CLASSES = ["Cyst", "Normal", "Stone", "Tumor"]
MODEL_PATH = "kidney_model_improved/final_kidney_model.keras"  # Path to the trained model
IMG_SIZE = (224, 224)

## 1. Improved Model Architecture

Let's examine the architecture of the improved model:

In [ ]:
# Code to build the improved model (for demonstration)
def build_improved_model():
    """Build and return the improved model architecture"""
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras import layers, models
    
    # Base model with pre-trained weights
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model
    base_model.trainable = False
    
    # Build model with proper regularization
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    
    # Strong regularization to prevent overfitting
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)  # High dropout rate
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)  # High dropout rate
    outputs = layers.Dense(4, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name="KidneyCNN_Improved")
    return model

# Build model for demonstration
improved_model = build_improved_model()
improved_model.summary()

## 2. Improved Data Augmentation

We've enhanced the data augmentation specifically for CT scans:

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Original data augmentation
original_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

# Improved data augmentation
improved_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,        # CT scans shouldn't be rotated too much
    zoom_range=0.1,           # Subtle zoom
    width_shift_range=0.05,   # Small shifts
    height_shift_range=0.05,  # Small shifts
    horizontal_flip=True,     # OK for kidneys
    brightness_range=[0.9, 1.1],  # Subtle brightness changes
    validation_split=0.2
)

# Visualize data augmentation
def visualize_augmentation(datagen, title):
    # Load a sample image
    sample_class = random.choice(CLASSES)
    sample_folder = os.path.join(BASE_PATH, sample_class)
    sample_file = random.choice(os.listdir(sample_folder))
    sample_path = os.path.join(sample_folder, sample_file)
    
    # Load image
    img = Image.open(sample_path)
    x = np.array(img.resize(IMG_SIZE))
    x = np.expand_dims(x, axis=0)
    
    # Create iterator
    it = datagen.flow(x, batch_size=1)
    
    # Plot original and augmented images
    plt.figure(figsize=(12, 3))
    plt.subplot(1, 5, 1)
    plt.imshow(img)
    plt.title('Original')
    plt.axis('off')
    
    for i in range(4):
        batch = it.next()
        aug_img = batch[0].astype('uint8')
        plt.subplot(1, 5, i+2)
        plt.imshow(aug_img)
        plt.title(f'Augmented {i+1}')
        plt.axis('off')
    
    plt.suptitle(f'{title} (Class: {sample_class})')
    plt.tight_layout()
    plt.show()

# Visualize both augmentation approaches
visualize_augmentation(original_datagen, 'Original Augmentation')
visualize_augmentation(improved_datagen, 'Improved Augmentation')

## 3. Improved Callbacks & Monitoring

We've changed the monitoring from `val_accuracy` to `val_loss` which provides better model selection:

In [ ]:
from tensorflow.keras import callbacks

# Original callbacks (problematic)
original_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        'best_cnn_kidney.h5', monitor='val_accuracy', save_best_only=True
    )
]

# Improved callbacks
improved_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',  # CRITICAL change
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        'best_cnn_kidney.keras',
        monitor='val_loss',  # CRITICAL change
        save_best_only=True
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

## 4. Fine-tuning Strategy

We've implemented a two-phase training approach with fine-tuning:

In [ ]:
# Phase 1: Train with frozen base model
# (Code just for demonstration, not for execution)
'''
# Initial training phase
model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=improved_callbacks
)
'''

# Phase 2: Fine-tuning - unfreeze later layers in base model
'''
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False
    
# Compile with a MUCH lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training with fine-tuning
model.fit(
    train_gen,
    epochs=EPOCHS,
    initial_epoch=history.epoch[-1],
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_list
)
'''

## 5. Clinical Recommendation System

The improved model still works with our clinical recommendation system:

In [ ]:
# Clinical recommendation system based on confidence levels
RECOMMENDATION_MAP = {
    'Cyst':   {
        'high':   "🔵 Kyste confirmé — Suivi échographique périodique (6 mois). Pas d'urgence.",
        'medium': "🟡 Kyste probable — Examen complémentaire recommandé (IRM ou échographie).",
        'low':    "⚠️ Diagnostic incertain — Consultation radiologique approfondie nécessaire."
    },
    'Normal': {
        'high':   "✅ Rein normal — Suivi de routine annuel. Aucune intervention requise.",
        'medium': "🟡 Probablement normal — Recontrôler dans 6 mois.",
        'low':    "⚠️ Incertitude diagnostique — Avis médical requis."
    },
    'Tumor':  {
        'high':   "🔴 URGENT — Tumeur détectée avec forte probabilité. Biopsie et consultation oncologique immédiate.",
        'medium': "🟠 Tumeur suspectée — Imagerie complémentaire (PET-scan) + avis oncologique urgent.",
        'low':    "⚠️ Diagnostic incertain — Examen multi-disciplinaire indispensable."
    },
    'Stone':  {
        'high':   "🟤 Calcul rénal confirmé — Analgésie + traitement selon taille (lithotripsie ou chirurgie).",
        'medium': "🟡 Calcul probable — Hydratation intensive + AINS + bilan urinaire.",
        'low':    "⚠️ Calcul possible — Échographie de confirmation recommandée."
    }
}

def generate_recommendation(image_path, model, threshold_high=0.80, threshold_mid=0.55):
    """Generate a clinical recommendation based on model prediction"""
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Get prediction
    probas = model.predict(img_array, verbose=0)[0]
    pred_idx = np.argmax(probas)
    pred_class = CLASSES[pred_idx]
    confidence = probas[pred_idx]
    
    # Determine confidence level
    if confidence >= threshold_high:
        level = 'high'
    elif confidence >= threshold_mid:
        level = 'medium'
    else:
        level = 'low'
    
    # Get recommendation
    reco = RECOMMENDATION_MAP[pred_class][level]
    
    # Print detailed report
    print("=" * 60)
    print(f"📋 DIAGNOSTIC REPORT")
    print("=" * 60)
    print(f"Predicted class: {pred_class}")
    print(f"Confidence     : {confidence:.1%}  [{level.upper()}]")
    print("\nSoftmax probabilities:")
    for cls, p in zip(CLASSES, probas):
        bar = '█' * int(p * 30)
        print(f"  {cls:<8}: {bar:<30} {p:.1%}")
    print(f"\n💊 Clinical recommendation:")
    print(f"   {reco}")
    print("=" * 60)
    
    # Show image
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class} ({confidence:.1%})")
    plt.axis('off')
    plt.show()
    
    return pred_class, confidence, level, reco

## 6. Using the Improved Model

Let's load our improved model and test it on a few random images:

In [ ]:
# Note: You'll need to have the trained model ready
# If you don't have the trained model yet, comment out this cell
# and run the training script first

'''
# Load the trained model
model = load_model(MODEL_PATH)

# Test on a random image from each class
for cls in CLASSES:
    # Get random image
    folder = os.path.join(BASE_PATH, cls)
    files = os.listdir(folder)
    test_file = random.choice(files)
    test_path = os.path.join(folder, test_file)
    
    print(f"\n\nTesting {cls} image: {test_file}")
    generate_recommendation(test_path, model)
'''

## 7. Expected Results & Benefits

The improved model should achieve:

1. **Higher Validation Accuracy**: 75-85% (compared to ~60-65%)
2. **Less Overfitting**: Training and validation curves should be closer
3. **Better Generalization**: More reliable predictions on unseen data
4. **Faster Convergence**: Fine-tuning helps reach better results quicker

### Comparison of Learning Curves

![Learning Curves Comparison](https://raw.githubusercontent.com/yourusername/kidney-classification/main/outputs/learning_curves_comparison.png)

## 8. Conclusion

By implementing these improvements, we've addressed the main issues in the original model:

1. **✅ Fixed Overfitting**: Transfer learning + proper regularization
2. **✅ Enhanced Augmentation**: CT-specific transformations
3. **✅ Better Monitoring**: Using val_loss instead of val_accuracy
4. **✅ Improved Generalization**: Fine-tuning strategy

These changes should result in a much more reliable kidney disease classification model that can be used with greater confidence in clinical settings.